# Unit Waveform Clustering

Analyze the spike waveforms of units in the **ventral pallidum (VP)** and use **unsupervised clustering** to test whether distinct populations (e.g. putative glutamatergic vs. GABAergic neurons) can be separated based on waveform shape.

In the Allen CCF the VP is covered by the basal-forebrain acronyms **SI** (substantia innominata) and **MA** (magnocellular preoptic nucleus), so units from both are pooled as VP.

**Workflow**
1. Load an ephys session and append CCF unit locations.
2. Select units in the target region(s).
3. For each unit, find the peak channel (largest trough-to-peak amplitude) and extract its mean waveform.
4. Extract shape features (trough-to-peak duration, half-width, peak/trough ratio, repolarization slope, ...).
5. Standardize features and estimate the number of clusters (elbow + silhouette).
6. Cluster with KMeans / Gaussian Mixture and visualize the resulting populations.

> Motivation: negative vs. positive correlation to value was observed for glutamatergic and GABAergic neurons in ventral pallidum — this notebook checks whether the two populations are separable purely from waveform morphology.

> Reusable helpers live in `waveform_clustering.py`; import them from any other notebook to analyze an arbitrary set of units (for example, opto-tagged neurons — see the last section).

## 1. Imports

In [ ]:
# =============================================================================
# 1. ENVIRONMENT SETUP & MODULE IMPORTS
# =============================================================================
"""
Initialize the analysis environment by loading essential modules and setting up
the Python path to access custom analysis functions.
"""

# Enable automatic reloading of modules for interactive development
%load_ext autoreload
%autoreload 2

# Import essential system modules
import sys
from pathlib import Path

# Define the path to custom analysis modules
# Note: Update this path to match your local installation
MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")

# Add module path to system path for importing custom functions
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"Analysis modules loaded from: {MODULE_PATH}")
print("Auto-reload enabled for interactive development")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

from sklearn.decomposition import PCA

# Project utilities (src/aind_dft_ephys_analysis)
from general_utils import find_ephys_sessions

# Reusable waveform-clustering functions (see waveform_clustering.py)
from waveform_clustering import (
    FEATURE_COLS,
    make_time_axis,
    normalize_waveform,
    collect_region_waveforms_from_sessions,
    compute_features_table,
    scale_features,
    estimate_n_clusters,
    cluster_waveforms,
    save_features_csv,
    analyze_units_waveforms,
)

%matplotlib inline

# ---- Configuration -------------------------------------------------------
# Waveform sampling rate (Neuropixels standard). Adjust if your probe differs.
SAMPLING_RATE_HZ = 30_000.0

# Brain regions to analyze. Ventral pallidum (VP) is covered by the
# basal-forebrain CCF acronyms SI (substantia innominata) and MA
# (magnocellular preoptic nucleus).
TARGET_REGIONS = ["SI", "MA"]

# Fixed window (in ms) extracted around each unit's trough so that waveforms
# from probes/sessions with different sample counts can be pooled together.
PRE_MS = 1.0
POST_MS = 2.0

# Where to write CSV outputs (unit properties, with session name embedded).
RESULTS_DIR = Path("/root/capsule/scratch/waveform_clustering")

RANDOM_STATE = 0

# Trough-aligned time axis / window sizes derived from the config above.
time_ms, PRE_SAMPLES, POST_SAMPLES, WIN_LEN = make_time_axis(
    PRE_MS, POST_MS, SAMPLING_RATE_HZ
)
feature_cols = FEATURE_COLS

## 2. Load all sessions and pool SI/MA units

Loop over every available (spike-sorted) session and, for each unit that (a) passes default QC and (b) is located in `TARGET_REGIONS`, extract the peak-channel waveform.

All the per-session loading / QC / trough-aligned extraction now lives in `waveform_clustering.collect_region_waveforms_from_sessions`. For speed it loads **only the ephys NWB** (`read_ephys_nwb`) and aligns each waveform to its trough over a fixed window (`PRE_MS` before, `POST_MS` after) so units from probes/sessions with different sample counts can be pooled.

In [ ]:
# Discover all ephys sessions (prefer spike-sorted ones)
all_sessions, sessions_by_animal, spike_sorted_sessions = find_ephys_sessions()
sessions_to_use = spike_sorted_sessions if len(spike_sorted_sessions) else all_sessions
print(f"Found {len(all_sessions)} sessions; using {len(sessions_to_use)} spike-sorted sessions.")

# Pool QC-passing, region-matched peak waveforms across every session.
pooled = collect_region_waveforms_from_sessions(
    sessions_to_use,
    target_regions=TARGET_REGIONS,
    pre_ms=PRE_MS,
    post_ms=POST_MS,
    sampling_rate_hz=SAMPLING_RATE_HZ,
    qc_only=True,
    verbose=True,
)

peak_waveforms = pooled["waveforms"]
kept_sessions = pooled["sessions"]
kept_indices = pooled["unit_indices"]
kept_regions = pooled["regions"]
regions_seen = pooled["regions_seen"]
time_ms = pooled["time_ms"]

# Diagnostic: if nothing matched, show which regions QC units actually had
# (helps catch acronym mismatches or missing CCF alignment).
print("\nRegions among QC-passing units (top 25):")
for region, count in regions_seen.most_common(25):
    print(f"  {region!r}: {count}")

if len(peak_waveforms) == 0:
    raise RuntimeError(
        f"No units matched {TARGET_REGIONS}. See the region list above: if regions "
        "are None/empty, the IBL_alignment (CCF) data was not found for these "
        "sessions; if the acronyms differ, update TARGET_REGIONS accordingly."
    )

print(f"\nPooled {peak_waveforms.shape[0]} QC-passing units across sessions in {TARGET_REGIONS}.")
print(f"Window: {WIN_LEN} samples ({PRE_MS} ms pre + {POST_MS} ms post trough).")

## 3. Overview of the pooled units

In [ ]:
print("Pooled units per region:")
for region, count in Counter(kept_regions).most_common():
    print(f"  {region}: {count}")

print("\nPooled units per session:")
for sess, count in Counter(kept_sessions).most_common():
    print(f"  {sess}: {count}")

## 4. Inspect the pooled peak-channel waveforms

Normalize each waveform (baseline-subtracted, trough-oriented negative, amplitude-normalized) for shape comparison across sessions.

In [ ]:
# Amplitude-normalize each (already baseline-corrected, trough-oriented,
# trough-aligned) waveform so shapes are comparable across units/sessions.
norm_waveforms = np.vstack([normalize_waveform(w) for w in peak_waveforms])

# Quick look at all normalized waveforms
plt.figure(figsize=(8, 5))
for w in norm_waveforms:
    plt.plot(time_ms, w, color='lightgrey', linewidth=0.5)
plt.plot(time_ms, norm_waveforms.mean(axis=0), color='C3', linewidth=2, label='mean')
plt.xlabel('Time (ms)')
plt.ylabel('Normalized amplitude')
plt.title(f'Peak-channel waveforms ({norm_waveforms.shape[0]} units)')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Extract waveform shape features

Classic features used to separate narrow- vs. broad-spiking neurons:
- **trough_to_peak_ms**: time from the trough to the following positive peak
- **half_width_ms**: full width of the trough at half its minimum
- **peak_trough_ratio**: amplitude of the post-trough peak relative to the trough
- **pre_peak_ratio**: amplitude of a positive peak *before* the trough (captures triphasic waveforms)
- **pre_peak_ms**: time from that pre-trough peak to the trough
- **repolarization_slope**: slope just after the trough
- **recovery_slope**: slope after the post-trough peak

In [ ]:
# Compute morphology features for every pooled unit (see
# waveform_clustering.extract_features / compute_features_table). Metadata
# columns (session, unit_index, region) are attached for downstream analysis.
features, norm_waveforms = compute_features_table(
    peak_waveforms,
    time_ms,
    sampling_rate_hz=SAMPLING_RATE_HZ,
    sessions=kept_sessions,
    unit_indices=kept_indices,
    regions=kept_regions,
    normalize=True,
)
features.head()

In [ ]:
# feature_cols is provided by the module (waveform_clustering.FEATURE_COLS).

# Distribution of the classic separator: trough-to-peak duration
plt.figure(figsize=(7, 4))
plt.hist(features['trough_to_peak_ms'], bins=30, color='C0', alpha=0.8)
plt.xlabel('Trough-to-peak duration (ms)')
plt.ylabel('Unit count')
plt.title('Trough-to-peak duration distribution')
plt.tight_layout()
plt.show()

## 6. Standardize features and estimate the number of clusters

In [ ]:
# Standardize features, then use elbow + silhouette diagnostics to estimate k.
X_scaled, _ = scale_features(features, feature_cols)
diag = estimate_n_clusters(X_scaled, k_range=range(2, 8), random_state=RANDOM_STATE)
ks, inertias, silhouettes, best_k = (
    diag["ks"], diag["inertias"], diag["silhouettes"], diag["best_k"]
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ks, inertias, 'o-')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow')
axes[1].plot(ks, silhouettes, 'o-', color='C1')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette'); axes[1].set_title('Silhouette score')
plt.tight_layout()
plt.show()

print(f"Best k by silhouette: {best_k}")

## 7. Cluster the units

We fit both KMeans and a Gaussian Mixture Model. Set `N_CLUSTERS` from the diagnostics above (defaults to the silhouette-optimal `best_k`, expecting at least 2 populations).

In [ ]:
N_CLUSTERS = max(3, best_k)  # override manually if desired

# Fit KMeans + GMM on the standardized features; adds cluster_kmeans / cluster_gmm.
clustering = cluster_waveforms(
    features, n_clusters=N_CLUSTERS, feature_cols=feature_cols, random_state=RANDOM_STATE
)
kmeans = clustering["kmeans"]
gmm = clustering["gmm"]
X_scaled = clustering["X_scaled"]

# Use GMM labels downstream (switch to 'cluster_kmeans' if preferred)
labels = features['cluster_gmm'].values
print(f"KMeans cluster sizes: {np.bincount(features['cluster_kmeans'].values)}")
print(f"GMM cluster sizes:    {np.bincount(features['cluster_gmm'].values)}")

## 8. Visualize the clusters

In [ ]:
# 8a. Mean waveform per cluster
plt.figure(figsize=(8, 5))
for c in sorted(np.unique(labels)):
    mask = labels == c
    mean_wf = norm_waveforms[mask].mean(axis=0)
    plt.plot(time_ms, mean_wf, linewidth=2, label=f'Cluster {c} (n={mask.sum()})')
    for w in norm_waveforms[mask]:
        plt.plot(time_ms, w, color=f'C{c}', alpha=0.08, linewidth=0.5)
plt.xlabel('Time (ms)')
plt.ylabel('Normalized amplitude')
plt.title('Mean waveform per cluster')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 8b. Feature space: trough-to-peak vs half-width, colored by cluster
plt.figure(figsize=(7, 6))
sc = plt.scatter(features['trough_to_peak_ms'], features['half_width_ms'],
                 c=labels, cmap='tab10', s=30, alpha=0.8)
plt.xlabel('Trough-to-peak duration (ms)')
plt.ylabel('Half-width (ms)')
plt.title('Waveform feature space by cluster')
plt.colorbar(sc, label='Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# 8c. PCA projection of the standardized feature space
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(7, 6))
sc = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', s=30, alpha=0.8)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
plt.title('PCA of waveform features by cluster')
plt.colorbar(sc, label='Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# 8d. Cluster composition by brain region
composition = pd.crosstab(features['region'], features['cluster_kmeans'])
print(composition)

composition.plot(kind='bar', stacked=True, figsize=(7, 4), colormap='tab10')
plt.xlabel('Brain region')
plt.ylabel('Unit count')
plt.title('Cluster composition per region')
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# 8e. Mean feature values per cluster (interpretation aid)
summary = features.groupby('cluster_kmeans')[feature_cols].mean()
summary['n_units'] = features.groupby('cluster_kmeans').size()
summary

## 9. Save unit properties to CSV

Persist the per-unit waveform features, cluster labels and metadata to a CSV. Every row already carries its own `session`, and a `session_name` label is stamped on the file. The filename embeds a label so repeated runs don't overwrite each other.

In [ ]:
# Save the pooled per-unit properties (features + cluster labels + metadata).
# Each row already carries its own `session`; we also stamp a run label.
csv_path = save_features_csv(
    features,
    out_dir=RESULTS_DIR,
    session_name="VP_SI_MA_pooled",
    suffix="_waveform_clustering.csv",
)
print(f"Saved {len(features)} units to: {csv_path}")
features.head()

## 10. Reuse: analyze the waveform of tagged neurons

The functions above live in `waveform_clustering.py`, so any other notebook can reuse them. Below we load a single session, take a set of **opto-tagged** unit indices, extract their peak-channel waveforms + morphology features, and save a CSV (with the session name embedded).

Replace `tagged_units` with any list of unit indices you want to inspect.

In [ ]:
# Reuse the module on an arbitrary set of units (here: opto-tagged neurons).
from nwb_utils import NWBUtils
from ephys_utils import append_units_locations
from general_utils import extract_session_name_core

TAGGED_SESSION = "ecephys_839480_2026-06-03_15-09-14_sorted-bandpass_2026-07-18_00-07-17"

nwb_tagged = NWBUtils.read_ephys_nwb(session_name=TAGGED_SESSION)
session_core = extract_session_name_core(TAGGED_SESSION) or TAGGED_SESSION
nwb_tagged = append_units_locations(nwb_tagged, session_name=session_core)

# Unit indices to analyze. If the opto-tagging columns were appended to the units
# table you can pull the tagged units automatically; otherwise pass any list.
try:
    from optotagging_Anna_nwb_export import select_tagged_units
    tagged_units = select_tagged_units(nwb_tagged)
except Exception as e:
    print(f"select_tagged_units unavailable ({e}); using a manual unit list.")
    tagged_units = [0, 1, 2]

print(f"{len(tagged_units)} units to analyze in {session_core}")

tagged = analyze_units_waveforms(
    nwb_tagged,
    unit_indices=tagged_units,
    session_name=session_core,
    pre_ms=PRE_MS,
    post_ms=POST_MS,
    sampling_rate_hz=SAMPLING_RATE_HZ,
    out_dir=RESULTS_DIR,
)
print(f"Saved tagged-unit properties to: {tagged['csv_path']}")
tagged["features"].head()

In [ ]:
# Inspect the tagged-unit waveforms
tw = tagged["norm_waveforms"]
tm = tagged["time_ms"]
plt.figure(figsize=(8, 5))
for w in tw:
    plt.plot(tm, w, color="lightgrey", linewidth=0.5)
if len(tw):
    plt.plot(tm, tw.mean(axis=0), color="C3", linewidth=2, label="mean")
plt.xlabel("Time (ms)")
plt.ylabel("Normalized amplitude")
plt.title(f"Tagged-unit waveforms (n={len(tw)})")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Single-neuron raster / PSTH per cluster

Open the pre-computed raster/PSTH PNGs for example units in each cluster to visually inspect their trial-by-trial activity. Figures are expected at:

`<base_dir>/ecephys_<session>_sorted_*/<model_latent>/<model_latent>_unit_<unit>.png`

In [ ]:
from PIL import Image
from IPython.display import display

# ---- Configuration -------------------------------------------------------
BASE_DIR = Path("/root/capsule/scratch/raster_plot")
MODEL_LATENT = "ForagingCompareThreshold-value-1"


N_PER_CLUSTER = 100          # example units to show per cluster (None -> all)
DISPLAY_WIDTH = 800        # px


def show_unit_figure(session_core, unit_id, model_latent=MODEL_LATENT):
    """Locate and display the saved raster/PSTH PNG for one unit. Returns True if shown."""
    session_folders = list(BASE_DIR.glob(f"ecephys_{session_core}_sorted_*"))
    if not session_folders:
        print(f"  no session folder for {session_core}")
        return False
    # Figures live in a per-latent subfolder: <session>/<latent>/<latent>_unit_<unit>.png
    fig_path = session_folders[0] / model_latent / f"{model_latent}_unit_{unit_id}.png"
    if not fig_path.exists():
        print(f"  missing: {fig_path}")
        return False
    img = Image.open(fig_path)
    w, h = img.size
    img = img.resize((DISPLAY_WIDTH, int(h * DISPLAY_WIDTH / w)), Image.LANCZOS)
    display(img)
    return True


# ---- Loop over clusters, show example units ------------------------------
for c in [2]:  # in sorted(features['cluster_kmeans'].unique()):
    members = features[features['cluster_kmeans'] == c]
    if N_PER_CLUSTER is not None:
        members = members.sample(min(N_PER_CLUSTER, len(members)), random_state=RANDOM_STATE)

    print("\n" + "=" * 50)
    print(f" Cluster {c}  ({(features['cluster_kmeans'] == c).sum()} units total, "
          f"showing {len(members)})")
    print("=" * 50)

    for _, row in members.iterrows():
        print(f"\n-- session {row['session']}, unit {row['unit_index']}, region {row['region']}")
        show_unit_figure(row['session'], int(row['unit_index']))

## 12. Notes & next steps

- Clusters with **short trough-to-peak / narrow half-width** are typically fast-spiking (often putative GABAergic); **broad** waveforms are typically putative glutamatergic. Compare `summary` above against this expectation.
- To test the value-correlation hypothesis, cross-reference the `unit_index` in each cluster with your value-encoding analysis and check whether negative- vs. positive-correlated units segregate by cluster.
- Consider pooling waveforms across multiple sessions for a larger, more robust sample before drawing conclusions.
- `SAMPLING_RATE_HZ` is set to 30 kHz — confirm it matches your recording, since all duration features scale with it.